# Tutorial 09: Human-in-the-Loop & Interactive Workflows

##  Learning Objectives
After completing this notebook, you will be able to:
- Implement **approval gates** that pause workflows for human review
- Create **interactive executors** that request user input
- Build **multi-turn conversations** with human feedback loops
- Use **context variables** to track approval state
- Implement **rejection handlers** for denied requests
- Understand **when human oversight is critical vs optional**

##  Prerequisites

Before starting this tutorial, you should have completed:
- **Tutorial 05: Multi-Agent Workflows** (Sequential, Concurrent patterns)
- **Tutorial 07: Advanced Workflows** (WorkflowBuilder, CustomExecutor)

##  What is Human-in-the-Loop?

**Human-in-the-Loop (HITL)** workflows pause execution to get human input:
-  **Approval Gates** - Humans review and approve/reject AI decisions
-  **Interactive Feedback** - Humans provide input to guide execution
-  **Quality Control** - Humans validate critical outputs
-  **Policy Compliance** - Humans ensure adherence to rules

### Why Use HITL

| Scenario | Reason for HITL |
|----------|----------------|
| Financial transactions | Legal requirement for approval |
| Content publishing | Quality and brand safety |
| Data deletion | Irreversible action |
| Customer communication | Reputation risk |
| Policy violations | Manual judgment needed |
| High-value decisions | Cost/risk mitigation |

### HITL Patterns in This Tutorial

1. **Approval Gate Pattern** - Simple approve/reject decision
2. **Interactive Refinement** - Multi-turn conversation with feedback
3. **Conditional Approval** - Different paths based on decision
4. **Budget-Based Approval** - Auto-approve below threshold, human needed otherwise

---

## Step 1: Setup and Imports

In [ ]:
import asyncio
from typing import cast


from agent_framework import (
    # Workflow builders
    WorkflowBuilder,
    # Core components
    Executor,
    AgentExecutor,
    AgentExecutorResponse,
    handler,
    WorkflowContext,
    # Events
    WorkflowEvent,
    AgentResponseUpdate,
    # Message types
    Message,
)
from agent_framework.orchestrations import SequentialBuilder
from agent_framework.azure import AzureAIAgentClient
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity.aio import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()
print(" Imports successful!")
print("\ud83c\udfad Ready to build Human-in-the-Loop workflows!")

In [ ]:
import os
mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === Authentication Method Selection ===
# True: API Key authentication, False: DefaultAzureCredential authentication
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("\ud83d\udd11 Authentication method: API Key authentication")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # The framework automatically reads the AZURE_OPENAI_API_KEY environment variable
    # and prioritizes it over the credential, so we explicitly remove it
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("\ud83d\udd10 Authentication method: DefaultAzureCredential")

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

In [ ]:
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import DefaultAzureCredential
from azure.identity.aio import AzureCliCredential, DefaultAzureCredential

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

def create_azure_openai_chat_client():
    """Create an Azure OpenAI client"""
    return AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )

In [ ]:
from agent_framework import WorkflowViz
from IPython.display import SVG, display
import os

def visualize_workflow(workflow, filename="workflow_diagram", show_preview=True):
    """
    Output workflow graph information, save as SVG, and preview.
    
    Args:
        workflow: The workflow object to visualize
        filename: The filename to save (without extension)
        show_preview: Whether to show the preview
    
    Returns:
        The path of the saved SVG file
    """
    # Create WorkflowViz object
    viz = WorkflowViz(workflow)
    
    # 3. Save as SVG file
    try:
        svg_path = viz.export(format="svg", filename=filename)
        print("=" * 60)
        print(f"\u2705 SVG file saved: {svg_path}")
        print("=" * 60)
        print()
        
        # 4. Preview the SVG
        if show_preview and os.path.exists(svg_path):
            display(SVG(filename=svg_path))
        
        return svg_path
        
    except ImportError as e:
        print("\u274c Error: 'graphviz' package is not installed")
        print("Installation: pip install agent-framework[viz] --pre")
        print(f"Details: {e}")
        return None
    except Exception as e:
        print(f"\u274c An error occurred: {e}")
        return None

## Step 2: Pattern 1 - Simple Approval Gate

The simplest HITL pattern: AI generates something, human approves or rejects.

### Flow
```
User Request → AI Agent → Human Review → (Approve) → Execute
                                        → (Reject)  → Stop
```

### Key Concepts
- **Synchronous input()** - Pause workflow to get user input
- **Conditional logic** - Route based on approval decision
- **Context tracking** - Save approval state for downstream use

In [ ]:
async def simple_approval_gate_demo():
    """
    Demonstrate Simple Approval Gate Pattern:
    AI drafts email → Human approves/rejects → Send if approved
    """
    print("="*70)
    print("Pattern 1: Simple Approval Gate")
    print("="*70)
    print("\nFlow: Email Draft → Human Review → Send if Approved\n")
    
    # Create email drafting agent
    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )

    email_agent = chat_client.as_agent(
        instructions="""
        You are a professional email writer.
        Draft clear, concise, and professional emails based on the user's request.
        Keep emails brief (max 3-4 sentences).
        Always include a subject line.
        """,
        name="EmailDrafter",
    )
    
    # CustomExecutor for human approval
    class ApprovalGate(Executor):
        """Pause workflow for human approval/rejection"""
        
        def __init__(self):
            super().__init__(id="approval_gate")
        
        @handler
        async def review(self, response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorResponse, str]) -> None:
            """
            Show draft to human and get approval decision.
            """
            # AgentExecutor returns AgentExecutorResponse, so extract accordingly
            draft = "No content"
            if response and getattr(response, "agent_run_response", None) and getattr(response.agent_run_response, "text", None):
                draft = response.agent_run_response.text
            elif response and getattr(response, "agent_response", None) and getattr(response.agent_response, "text", None):
                draft = response.agent_response.text
            
            print("\n" + "="*70)
            print("\ud83d\udce7 Email Draft for Review")
            print("="*70)
            print(draft)
            print("="*70)
            
            # Pause workflow for human input
            while True:
                decision = input("\n\ud83d\udc64 Do you approve this email? (yes/no): ").strip().lower()
                
                if decision in ['yes', 'y']:
                    print(" Email approved!")
                    await ctx.yield_output(f" Approved and sent\n\n{draft}")
                    break
                elif decision in ['no', 'n']:
                    print(" Email rejected!")
                    await ctx.yield_output(" Rejected - Email was not sent")
                    break
                else:
                    print("  Please enter 'yes' or 'no'")
    
    # Build workflow: Agent → Approval Gate
    email_executor = AgentExecutor(email_agent, id="EmailDrafter")
    approval_gate = ApprovalGate()
    
    workflow = (
        SequentialBuilder(participants=[email_executor, approval_gate])
        .build()
    )
    
    # Visualize the workflow
    svg_path = visualize_workflow(
        workflow, 
        filename="simple_approval_gate_workflow.svg",
        show_preview=True
    )

    # Test request
    request = "Create an email announcing the quarterly all-hands meeting to the team, scheduled for next Friday at 2 PM."
    
    print(f"\n Request: {request}\n"
          )
    print("\ud83e\udd16 AI is drafting the email...\n")
    
    # Run the workflow
    final_output = None
    async for event in workflow.run(request, stream=True):
        if isinstance(event, WorkflowEvent) and event.type == "output":
            if not isinstance(event.data, AgentResponseUpdate):
                final_output = event.data
    
    # Display final result
    print("\n" + "="*70)
    print(" Final Result")
    print("="*70)
    if final_output:
        print(final_output)
    print("="*70)
    
    print("\n What happened:")
    print("  1, AI drafted the email")
    print("  2, Workflow paused for human review")
    print("  3, Human approved or rejected")
    print("  4, Workflow completed based on decision")

await simple_approval_gate_demo()


This pattern verifies **"whether a custom approval gate using `Executor` can insert human judgment in the middle of a Sequential workflow"**.

Specific verification points:

1. **HITL implementation via `Executor` subclass** — Defining a `@handler` in `ApprovalGate(Executor)` and synchronously blocking workflow execution with `input()` to wait for human input
2. **Sequential chaining of `AgentExecutor` → custom `Executor`** — Using `SequentialBuilder().participants([email_executor, approval_gate])` so the AI agent's output (`AgentExecutorResponse`) is passed directly as the handler argument to the next Executor
3. **Branching output via `ctx.yield_output()`** — Dynamically switching the workflow's final output based on the approval/rejection decision
4. **Receiving the preceding output type** — Safely extracting text from `AgentExecutorResponse` using `getattr` to access `agent_run_response.text` / `agent_response.text`

This is a working demonstration that **the simplest HITL pattern (AI generation → Human Yes/No → Result determination) can be achieved with `SequentialBuilder` + custom `Executor`**.

## Step 3: Pattern 2 - Interactive Refinement Loop

More sophisticated: allow humans to provide feedback and iterate.

### Flow
```
Request → AI Draft → Human Review ┬→ Approve → Done
                                   └→ Feedback → AI Revise → Human Review (Loop)
```

### Key Concepts
- **Multi-turn conversation** - Iterative improvement
- **Feedback integration** - Human guides AI refinement
- **Loop control** - Termination conditions (approval or max iterations)

In [ ]:
async def interactive_refinement_demo():
    """
    Demonstrate Interactive Refinement:
    AI drafts → Human provides feedback → AI revises → Repeat until approved
    """
    print("="*70)
    print("Pattern 2: Interactive Refinement Loop")
    print("="*70)
    print("\nFlow: Draft → Review → Feedback → Revise → Repeat\n")
    
    # Create content writer agent
    # credential = AzureCliCredential()
    # chat_client = AzureAIAgentClient(async_credential=credential)
    
    # Create Azure OpenAI client
    # chat_client = AzureOpenAIChatClient(
    #     api_key=azure_openai_key,
    #     endpoint=azure_openai_endpoint,
    #     api_version=api_version,
    #     deployment_name=azure_deployment,
    # )

    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )

    writer_agent = chat_client.as_agent(
        instructions="""
        You are a creative blog post writer.
        Write engaging and informative blog posts on the requested topic.
        When feedback is provided, revise the post to reflect it.
        Keep posts concise (3-4 paragraphs).
        """,
        name="BlogWriter",
    )
    
    # Interactive Refinement Executor
    class InteractiveEditor(Executor):
        """Allow humans to iteratively refine AI output"""
        
        def __init__(self, agent, max_iterations=3):
            super().__init__(id="interactive_editor")
            self.agent = agent
            self.max_iterations = max_iterations
        
        @handler
        async def edit_loop(self, request: str, ctx: WorkflowContext[str, str]) -> None:
            """
            Interactive editing loop with human feedback.
            """
            conversation_history = [Message(role="user", text=request)]
            iteration = 0
            
            while iteration < self.max_iterations:
                iteration += 1
                print(f"\n{'='*70}")
                print(f" Draft {iteration} (max {self.max_iterations})")
                print("="*70)
                
                # Get AI response
                response = await self.agent.run(conversation_history)
                draft = response.text
                
                print(draft)
                print("="*70)
                
                # Get human feedback
                decision = input("\n\ud83d\udc64 (a)pprove, (f)eedback, or (r)eject? ").strip().lower()
                
                if decision in ['a', 'approve']:
                    print("\n Content approved!")
                    await ctx.yield_output(f" Approved (after {iteration} iteration(s))\n\n{draft}")
                    return
                
                elif decision in ['r', 'reject']:
                    print("\n Content rejected!")
                    await ctx.yield_output(" Rejected - Content was not published")
                    return
                
                elif decision in ['f', 'feedback']:
                    feedback = input("\n What changes would you like? ")
                    print(f"\n Feedback recorded: {feedback}")
                    print(" AI is revising based on feedback...")
                    
                    # Append to conversation history
                    conversation_history.append(Message(role="assistant", text=draft))
                    conversation_history.append(
                        Message(
                            role="user", 
                            text=f"Please revise based on this feedback: {feedback}"
                        )
                    )
                else:
                    print("Invalid option. Treating as a feedback request.")
                    feedback = input("\n What changes would you like? ")
                    conversation_history.append(Message(role="assistant", text=draft))
                    conversation_history.append(
                        Message(
                            role="user", 
                            text=f"Please revise based on this feedback: {feedback}"
                        )
                    )
            
            # Max iterations reached
            print(f"\n  Maximum iterations ({self.max_iterations}) reached.")
            final_decision = input("\ud83d\udc64 Do you approve the final draft? (yes/no): ").strip().lower()
            
            if final_decision in ['yes', 'y']:
                await ctx.yield_output(f" Approved (max iterations reached)\n\n{draft}")
            else:
                await ctx.yield_output(" Rejected - Max iterations reached without approval")
    
    # Build workflow
    editor = InteractiveEditor(writer_agent, max_iterations=3)
    
    workflow = (
        WorkflowBuilder(start_executor=editor)
        .build()
    )
    # Visualize the workflow
    svg_path = visualize_workflow(
        workflow, 
        filename="interactive_refinement_workflow.svg",
        show_preview=True
    )

    # Test request
    request = "Write a blog post about the benefits of AI agents in customer service."
    
    print(f"\n Request: {request}\n")
    print("\ud83e\udd16 AI is creating the initial draft...\n")
    
    # Run the workflow
    final_output = None
    async for event in workflow.run(request, stream=True):
        if isinstance(event, WorkflowEvent) and event.type == "output":
            if not isinstance(event.data, AgentResponseUpdate):
                final_output = event.data
    
    # Display final result
    print("\n" + "="*70)
    print(" Final Result")
    print("="*70)
    if final_output:
        print(final_output)
    print("="*70)
    
    print("\n What happened:")
    print("  1, AI created the initial draft")
    print("  2, Human reviewed and provided feedback")
    print("  3, AI revised based on feedback")
    print("  4, Continues until approved or max iterations reached")
    print("  5, Multi-turn conversation maintains context")

await interactive_refinement_demo()

This pattern verifies **"whether a multi-turn iterative loop that takes human feedback and has AI regenerate can be achieved within a single `Executor`"**.

Specific verification points:

1. **HITL via self-managed loop inside an Executor** — The `@handler` in `InteractiveEditor(Executor)` has a `while` loop that uses `input()` to get a 3-way choice of approve(a) / feedback(f) / reject(r), enabling iterative control within the Executor alone without relying on the workflow engine
2. **Multi-turn via manual conversation history management** — Adding `Message(user/assistant)` to the `conversation_history` list each time and passing it to `agent.run(conversation_history)` to have AI generate revised versions incorporating feedback (self-managed history rather than using the framework's conversation management)
3. **Safe termination guarantee via `max_iterations`** — Even if the human keeps providing feedback indefinitely, `max_iterations` forcefully breaks the loop and requests a final decision
4. **`WorkflowBuilder().set_start_executor(editor).build()`** — A minimal single-node workflow configuration where the Executor operates as a standalone node rather than a Sequential chain

Compared to Pattern 1 (Approval Gate), this demonstrates that **an iterative improvement loop of "Feedback → AI Revision → Re-review" (not just a one-time Yes/No decision) can be implemented with nothing more than logic inside the Executor**.

## Step 4: Pattern 3 - Conditional Approval with Branching

Different workflow paths based on approval decision.

### Flow
```
Request → Planner Agent → Human Review ┬→ Approved → Execution Agent → Done
                                        └→ Rejected → Alternative Path → Done
```

### Key Concepts
- **Branching logic** - Different downstream executors
- **Context passing** - Share approval state
- **Fallback paths** - Handle rejections gracefully

In [ ]:
async def conditional_approval_demo():
    """
    Demonstrate Conditional Approval with Branching:
    Plan → Approve/Reject → Different paths based on decision
    """
    print("="*70)
    print("Pattern 3: Conditional Approval with Branching")
    print("="*70)
    print("\nFlow: Plan → Review → Branch (Approved/Rejected Path)\n")
    
    # Create planning agent
    # credential = AzureCliCredential()
    # chat_client = AzureAIAgentClient(async_credential=credential)
    
    # Create Azure OpenAI client
    # chat_client = AzureOpenAIChatClient(
    #     api_key=azure_openai_key,
    #     endpoint=azure_openai_endpoint,
    #     api_version=api_version,
    #     deployment_name=azure_deployment,
    # )

    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )
    planner_agent = chat_client.as_agent(
        instructions="""
        You are a project planning assistant.
        Create detailed project plans including timeline, budget, and resources.
        Keep plans concise (4-5 key points).
        """,
        name="ProjectPlanner",
    )
    
    execution_agent = chat_client.as_agent(
        instructions="""
        You are a project execution assistant.
        Based on the approved plan, create a step-by-step execution guide.
        Make it concrete and actionable.
        """,
        name="ExecutionPlanner",
    )
    
    alternative_agent = chat_client.as_agent(
        instructions="""
        You are a creative alternatives advisor.
        When the original plan is rejected, suggest alternative approaches.
        Provide 2-3 different options to consider.
        """,
        name="AlternativesAdvisor",
    )
    
    # Approval gate with branching
    class BranchingApprovalGate(Executor):
        """Approval gate that determines the next executor based on decision"""
        
        def __init__(self):
            super().__init__(id="branching_approval")
            self.approved = False
        
        @handler
        async def review_and_branch(
            self,
            response: AgentExecutorResponse,
            ctx: WorkflowContext[AgentExecutorResponse, str],
        ) -> None:
            """
            Review the plan and route to the appropriate next step.
            """
            # Extract text from AgentExecutorResponse (agent_response, not agent_run_response)
            plan = "No content"
            if response and getattr(response, "agent_response", None) and getattr(response.agent_response, "text", None):
                plan = response.agent_response.text
            
            print("\n" + "="*70)
            print("\ud83d\udccb Project Plan for Review")
            print("="*70)
            print(plan)
            print("="*70)
            
            # Get approval decision
            while True:
                decision = input("\n\ud83d\udc64 Do you approve this plan? (yes/no): ").strip().lower()
                
                if decision in ['yes', 'y']:
                    print(" Plan approved! Creating execution guide...")
                    self.approved = True
                    # Send with routing prefix (branching via edge condition)
                    await ctx.send_message(
                        "APPROVED:\n" + f"Create execution steps for the following plan:\n\n{plan}"
                    )
                    break
                elif decision in ['no', 'n']:
                    print("Plan rejected! Suggesting alternatives...")
                    self.approved = False
                    await ctx.send_message(
                        "REJECTED:\n" + f"The original plan was rejected. Suggest alternatives for the following plan:\n\n{plan}"
                    )
                    break
                else:
                    print("  Please enter 'yes' or 'no'")
    
    # Result handler
    class ResultHandler(Executor):
        """Format final output based on approval path"""
        
        def __init__(self, approval_gate):
            super().__init__(id="result_handler")
            self.approval_gate = approval_gate
        
        @handler
        async def format_result(
            self,
            response: AgentExecutorResponse,
            ctx: WorkflowContext[AgentExecutorResponse, str],
        ) -> None:
            # Extract content from response
            content = "No content"
            if response and getattr(response, "agent_response", None) and getattr(response.agent_response, "text", None):
                content = response.agent_response.text
            
            if self.approval_gate.approved:
                output = f" Plan Approved\n\n\ud83d\udccb Execution Guide:\n{content}"
            else:
                output = f" Plan Rejected\n\n Alternative Options:\n{content}"
            
            await ctx.yield_output(output)
    
    # Build workflow with branching
    planner_executor = AgentExecutor(planner_agent, id="ProjectPlanner")
    approval_gate = BranchingApprovalGate()
    execution_executor = AgentExecutor(execution_agent, id="ExecutionPlanner")
    alternative_executor = AgentExecutor(alternative_agent, id="AlternativesAdvisor")
    result_handler = ResultHandler(approval_gate)
    
    # Branching with condition-based edges (prefix-based routing on string messages)
    workflow = (
        WorkflowBuilder(start_executor=planner_executor)
        .add_edge(planner_executor, approval_gate)
        .add_edge(
            approval_gate,
            execution_executor,
            condition=lambda msg: isinstance(msg, str) and msg.startswith("APPROVED:"),
        )
        .add_edge(
            approval_gate,
            alternative_executor,
            condition=lambda msg: isinstance(msg, str) and msg.startswith("REJECTED:"),
        )
        .add_edge(execution_executor, result_handler)
        .add_edge(alternative_executor, result_handler)
        .build()
    )
    # Visualize the workflow
    svg_path = visualize_workflow(
        workflow,
        filename="conditional_approval_workflow.svg",
        show_preview=True
    )

    # Test request
    request = "Create a plan to launch a new mobile app in 3 months."
    
    print(f"\n Request: {request}\n")
    print("\ud83e\udd16 AI is creating the project plan...\n")
    
    # Run the workflow
    final_output = None
    async for event in workflow.run(request, stream=True):
        if isinstance(event, WorkflowEvent) and event.type == "output":
            if not isinstance(event.data, AgentResponseUpdate):
                final_output = event.data
    
    # Display final result
    print("\n" + "="*70)
    print(" Final Result")
    print("="*70)
    if final_output:
        print(final_output)
    print("="*70)
    
    print("\n What happened:")
    print("  1, AI created the initial project plan")
    print("  2, Human reviewed and approved/rejected")
    print("  3, Workflow branched based on decision")
    print("  4, Approved → Execution guide created")
    print("  5, Rejected → Alternative options suggested")

await conditional_approval_demo()

This pattern verifies **"whether the workflow can dynamically branch using `WorkflowBuilder`'s conditional edges (`add_edge` + `condition`) based on human approval/rejection decisions"**.

Specific verification points:

1. **Routing via `ctx.send_message()` + `condition` lambda** — The approval gate sends messages downstream using `ctx.send_message()` instead of `ctx.yield_output()`, and the edge's `condition=lambda msg: msg.startswith("APPROVED:")` / `"REJECTED:"` determines which path to follow
2. **DAG-type workflow construction** — Using `WorkflowBuilder` + `add_edge` instead of `SequentialBuilder` (linear) to build a diamond-shaped branching/merging graph: "Plan → Approval Gate → (if approved) Execution Agent / (if rejected) Alternatives Agent → Result Handler"
3. **State sharing between Executors** — `ResultHandler` references `self.approval_gate.approved` to switch the final output format, demonstrating cross-node state sharing via Executor instance fields
4. **Coordination of multiple `AgentExecutors` + multiple custom `Executors`** — Three AI agents (Planner / ExecutionPlanner / AlternativesAdvisor) and two custom Executors (BranchingApprovalGate / ResultHandler) working together in one workflow

Compared to Patterns 1 and 2, this demonstrates that **HITL decisions don't just toggle output content, but actually branch the subsequent workflow paths themselves (approved → execution plan, rejected → alternative suggestions routed to different AI agents)**.

## Step 5: Pattern 4 - Smart Approval (Auto-Approve vs Manual)

Automatically approve low-risk requests, require human approval for high-risk ones.

### Flow
```
Request → Risk Analysis ┬→ Low Risk  → Auto-Approve → Execute
                        └→ High Risk → Human Review → (Approve) → Execute
                                                     → (Reject)  → Stop
```

### Key Concepts
- **Risk assessment** - Business logic determines approval path
- **Efficiency** - Skip human review when safe
- **Conditional gate** - Only pause when needed

In [ ]:
async def smart_approval_demo():
    """
    Demonstrate Smart Approval:
    Low-value purchases auto-approved, high-value requires human approval
    """
    print("="*70)
    print("Pattern 4: Smart Approval (Auto vs Manual)")
    print("="*70)
    print("\nFlow: Analyze → Route (Auto-Approve / Human Review)\n")
    
    # Purchase recommendation agent
    # credential = AzureCliCredential()
    # chat_client = AzureAIAgentClient(async_credential=credential)
    
    # Create Azure OpenAI client
    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )
    purchase_agent = chat_client.as_agent(
        instructions="""
        You are a purchase assistant.
        Recommend specific items to purchase based on the request.
        Include estimated cost in your response.
        Format: "Item: [name], Estimated Cost: $[amount], Reason: [reason]"
        """,
        name="PurchaseAdvisor",
    )
    
    # Smart approval gate with threshold
    class SmartApprovalGate(Executor):
        """Auto-approve below threshold, require human approval above"""
        
        def __init__(self, threshold=100):
            super().__init__(id="smart_approval")
            self.threshold = threshold
        
        @handler
        async def smart_approve(
            self,
            response: AgentExecutorResponse,
            ctx: WorkflowContext[AgentExecutorResponse, str],
        ) -> None:
            """
            Extract cost and auto-approve or request human review.
            """
            # AgentExecutor returns AgentExecutorResponse, extract text accordingly
            recommendation = "No content"
            if response and getattr(response, "agent_response", None) and getattr(response.agent_response, "text", None):
                recommendation = response.agent_response.text
            elif response and getattr(response, "full_conversation", None):
                conv = response.full_conversation or []
                if conv and getattr(conv[-1], "text", None):
                    recommendation = conv[-1].text
            
            print("\n" + "="*70)
            print(" Purchase Recommendation")
            print("="*70)
            print(recommendation)
            print("="*70)
            
            # Extract cost (simplified - look for $amount)
            import re
            cost_match = re.search(r'\$([0-9,]+)', recommendation or "")
            
            if cost_match:
                cost_str = cost_match.group(1).replace(',', '')
                cost = float(cost_str)
                
                print(f"\n Detected cost: ${cost:.2f}")
                print(f"   Approval threshold: ${self.threshold:.2f}")
                
                if cost <= self.threshold:
                    print(f"\n Auto-approved (below ${self.threshold} threshold)")
                    await ctx.yield_output(
                        f" Auto-approved (${cost:.2f} \u2264 ${self.threshold:.2f})\n\n{recommendation}"
                    )
                else:
                    print(f"\n  Human approval required (above ${self.threshold} threshold)")
                    
                    # Pause for human review
                    while True:
                        decision = input("\n\ud83d\udc64 Do you approve this purchase? (yes/no): ").strip().lower()
                        
                        if decision in ['yes', 'y']:
                            print(" Approved by human!")
                            await ctx.yield_output(
                                f" Approved by human (${cost:.2f} > ${self.threshold:.2f})\n\n{recommendation}"
                            )
                            break
                        elif decision in ['no', 'n']:
                            print(" Purchase rejected!")
                            await ctx.yield_output(
                                f" Rejected (${cost:.2f} > ${self.threshold:.2f})\n\nReason: Rejected by approver"
                            )
                            break
                        else:
                            print("  Please enter 'yes' or 'no'")
            else:
                print("\n  Could not extract cost - human review required")
                decision = input("\n\ud83d\udc64 Do you approve this purchase? (yes/no): ").strip().lower()
                
                if decision in ['yes', 'y']:
                    await ctx.yield_output(f" Approved\n\n{recommendation}")
                else:
                    await ctx.yield_output(" Rejected")
    
    # Build workflow
    purchase_executor = AgentExecutor(purchase_agent, id="PurchaseAdvisor")
    smart_gate = SmartApprovalGate(threshold=500.00)  # Auto-approve below $500
    
    workflow = (
        SequentialBuilder(participants=[purchase_executor, smart_gate])
        .build()
    )
    
    # Visualize the workflow
    svg_path = visualize_workflow(
        workflow,
        filename="smart_approval_workflow.svg",
        show_preview=True
    )

    # Test with multiple scenarios
    test_cases = [
        "Recommend office supplies for the team (pens, notebooks, etc.)",
        "Recommend a new laptop for the development team lead",
    ]
    
    for i, request in enumerate(test_cases, 1):
        print("\n\n" + "#"*70)
        print(f"Test Case {i}")
        print("#"*70)
        print(f"\n Request: {request}\n")
        print("\ud83e\udd16 AI is analyzing the request...\n")
        
        final_output = None
        async for event in workflow.run(request, stream=True):
            if isinstance(event, WorkflowEvent) and event.type == "output":
                if not isinstance(event.data, AgentResponseUpdate):
                    final_output = event.data
        
        print("\n" + "="*70)
        print(" Result")
        print("="*70)
        if final_output:
            print(final_output)
        print("="*70)
    
    print("\n" + "="*70)
    print(" What happened:")
    print("="*70)
    print("  1, AI recommended purchases with cost estimates")
    print("  2, Smart gate analyzed cost vs threshold")
    print("  3, Low-cost items auto-approved instantly")
    print("  4, High-cost items required human review")
    print("  5, Efficiency: Skip review when safe, require when risky")
    print("\n Production use cases:")
    print("  \u2022 Financial transactions (auto-approve < $X)")
    print("  \u2022 Content moderation (auto-approve low risk)")
    print("  \u2022 Resource allocation (auto-approve standard requests)")
    print("  \u2022 Data access (auto-approve public, review sensitive)")

await smart_approval_demo()

This pattern verifies **"whether HITL can be dynamically triggered only when needed by programmatically evaluating AI output against business rules (thresholds)"**.

Specific verification points:

1. **Selective triggering of conditional HITL** — Rather than asking humans in every case, costs are extracted via regex and if `cost <= threshold`, `input()` is not called and auto-approval proceeds; human judgment is requested only when exceeded. The ability to toggle HITL invocation at runtime
2. **Programmatic parsing of AI output** — Using `re.search(r'\$([0-9,]+)', ...)` to extract structured data (amount) from the agent's natural language response and using it as input for business logic
3. **Multiple test case execution on the same workflow** — Running low-cost (office supplies) and high-cost (laptop) cases through the same `workflow`, demonstrating that the same approval gate exhibits different behavior (auto-approve vs manual approve) depending on input
4. **Fallback when cost cannot be extracted** — When the regex doesn't match, it falls back to the safe side by requiring human review, a defensive design

Compared to Patterns 1-3, this demonstrates that **a smart gate that triggers HITL "selectively based on risk/cost" rather than "always" can be achieved with nothing more than conditional logic inside the Executor**.

##  Key Takeaways

### What We Learned

1. **Approval Gates**
   - Use `input()` inside executors to pause workflows
   - Get human approval/rejection decisions
   - Route workflows based on decisions

2. **Interactive Refinement**
   - Multi-turn conversation with feedback
   - Maintain conversation history
   - Iterate until approved or max iterations reached

3. **Conditional Branching**
   - Different paths based on approval
   - Share state between executors
   - Handle rejections gracefully

4. **Smart Approval**
   - Business logic determines approval path
   - Auto-approve low-risk items
   - Human review for high-risk items
   - Balance efficiency and oversight

### HITL Best Practices

**When to use HITL:**
-  High-risk decisions (financial, legal)
-  Irreversible actions (deletion, publishing)
-  Compliance requirements (approval policies)
-  Quality control (brand safety)
-  Edge cases AI cannot handle

**When to skip HITL:**
-  High-volume, low-risk tasks
-  Well-defined routine operations
-  Real-time requirements
-  Development/testing environments

### Production Patterns

```python
# Simple Approval Gate
class ApprovalGate(Executor):
    @handler
    async def review(self, content: str, ctx: WorkflowContext[str, str]):
        decision = input("Approve? (yes/no): ")
        if decision == 'yes':
            await ctx.yield_output(f"Approved: {content}")
        else:
            await ctx.yield_output("Rejected")

# Smart Approval with Threshold
class SmartGate(Executor):
    def __init__(self, threshold):
        super().__init__(id="smart_gate")
        self.threshold = threshold
    
    @handler
    async def review(self, data: dict, ctx: WorkflowContext):
        if data['risk_score'] < self.threshold:
            await ctx.yield_output("Auto-approved")
        else:
            decision = input("High risk - approve? ")
            await ctx.yield_output(decision)
```

### Real-World Applications

1. **Content Publishing Pipeline**
   - AI drafts content → Editor reviews → Publish if approved

2. **Customer Support Escalation**
   - AI handles routine inquiries → Escalate complex ones to humans → Human provides guidance

3. **Financial Approval Workflow**
   - Auto-approve < $1000 → Manager approval $1000-$10k → Executive approval > $10k

4. **Data Access Control**
   - Auto-approve public data → Sensitive data requires approval

---

##  Practice Exercises

### Exercise 1: Multi-Level Approval

Create a workflow with tiered approval:
```
Request → Manager ($0-$5k) → Director ($5k-$25k) → CFO (>$25k)
```

**Hint:** Use nested approval gates with different thresholds.

### Exercise 2: Feedback Loop with Memory

Create an interactive system that:
- Remembers previous feedback
- Shows improvements per iteration
- Learns user preferences

**Hint:** Maintain and reference conversation history.

### Exercise 3: Approval Analytics

Track and report:
- Auto-approval rate
- Manual approval rate
- Average review time
- Rejection reasons

**Hint:** Add logging to the approval gate.

In [ ]:
# Exercise Playground - Implement your solutions here!

async def multi_level_approval_exercise():
    """
    Exercise 1: Build a tiered approval workflow.
    """
    # TODO: Implement multi-level approval
    pass

async def feedback_memory_exercise():
    """
    Exercise 2: Build a feedback loop with memory.
    """
    # TODO: Implement feedback loop with context
    pass

async def approval_analytics_exercise():
    """
    Exercise 3: Track approval metrics.
    """
    # TODO: Implement analytics tracking
    pass

print(" Exercise templates ready - Implement your solutions above!")

##  What's Next?

Congratulations! You've mastered Human-in-the-Loop patterns!

You can now:
-  Pause workflows for human approval
-  Create interactive refinement loops
-  Implement conditional branching based on decisions
-  Build smart approval gates with thresholds
-  Choose when human oversight is needed

---

### Quick Reference

**Approval Gate Pattern:**
```python
class ApprovalGate(Executor):
    def __init__(self):
        super().__init__(id="approval")
    
    @handler
    async def review(self, content: str, ctx: WorkflowContext[str, str]):
        print(f"Review: {content}")
        decision = input("Approve? (yes/no): ")
        
        if decision == 'yes':
            await ctx.yield_output(f"Approved: {content}")
        else:
            await ctx.yield_output("Rejected")
```

**Interactive Loop Pattern:**
```python
conversation = [Message(role="user", text=request)]
for i in range(max_iterations):
    response = await agent.run(conversation)
    feedback = input("Feedback: ")
    conversation.append(Message(role="assistant", text=response.text))
    conversation.append(Message(role="user", text=feedback))
```

**Smart Approval Pattern:**
```python
if risk_score < threshold:
    print("Auto-approved")
    await ctx.yield_output(content)
else:
    decision = input("High risk - approve? ")
    if decision == 'yes':
        await ctx.yield_output(content)
```

---

** Great job completing Tutorial 08!**